# Executorch Implementation

In [13]:
import torch
from typing import Optional, Any
from executorch.exir import EdgeProgramManager, ExecutorchBackendConfig, to_edge, to_edge_transform_and_lower
from executorch.exir.backend.backend_api import LoweredBackendModule, to_backend
from executorch.exir.passes import MemoryPlanningPass
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner

from src.utils import load_data
from src.Quantization.quantization_utils.model_setup import quantization_mode
from src.utils.model_setup import setup_model
from src.Quantization.quantization_utils.conversions.litert import quantize_pytorch_model

### Convert to ExecuTorch Program

In [2]:
def convert_to_executorch_program(
    model: torch.nn.Module,
    example_inputs: tuple[torch.Tensor, ...],
    save_path: Optional[str] = "model.pte",
    verbose: bool = False
):
    """
    Converts a PyTorch model into an ExecuTorch program and saves it as a .pte file.

    Parameters:
        model (nn.Module): The PyTorch model to convert.
        example_inputs (tuple[Tensor, ...]): Example inputs for tracing/export.
        save_path (Optional[str]): File path to save the .pte ExecuTorch program.

    Returns:
        ExecuTorchProgram: The compiled ExecuTorch program.
    """
    # model.eval()  # Ensure model is in eval mode for export

    # Step 1: Export the model to ATen dialect
    aten_dialect_graph = torch.export.export(model, example_inputs)

    if verbose:
        print("Aten Dialect Graph:")
        print(aten_dialect_graph)

    # Step 2: Apply optimizations for Edge devices
    # edge_program = to_edge(aten_dialect_graph)
    edge_program = to_edge_transform_and_lower(
        aten_dialect_graph,
        partitioner=[XnnpackPartitioner()],
        )

    if verbose:
        print("Edge Program Graph:")
        print(edge_program.exported_program())

    # Step 3: Convert to ExecuTorch program
    executorch_program = edge_program.to_executorch(
        ExecutorchBackendConfig(
        passes=[],  # User-defined passes
        memory_planning_pass=MemoryPlanningPass(),  # Default memory planning pass
        )
    )

    # Step 4: Save the ExecuTorch program as .pte
    if save_path:
        with open(save_path, "wb") as f:
            f.write(executorch_program.buffer)
        print(f"ExecuTorch program saved to {save_path}")

    return executorch_program

In [14]:
from executorch.exir import EdgeCompileConfig
from executorch.extension.export_util.utils import export_to_edge, save_pte_program
import os

def convert_to_edge(
        quantized_model: torch.nn.Module,
        example_inputs: tuple[torch.Tensor, ...],
        save_path: Optional[str] = "model.pte"
):
    
    try:
        _ = torch.ops.quantized_decomposed.add.out
    except AttributeError:
        print("No registered quantized ops")

    torch.ops.load_library("../executorch/cmake-out/kernels/quantized/libquantized_ops_aot_lib.so")
    edge_compile_config = EdgeCompileConfig(_check_ir_validity=False)
    edge_m = export_to_edge(
        quantized_model, example_inputs, edge_compile_config=edge_compile_config
    )

    prog = edge_m.to_executorch(
        config=ExecutorchBackendConfig(extract_delegate_segments=False)
    )

    filename = save_pte_program(prog, f"test_quantized")
    print("Executorch Size:", os.path.getsize(f"{filename}") / 1e6)

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pretrained_weights = f"models/SkinCancer/Quantized/mobilenet_v2_qat_kd.pth"
batch_size = 32
dataloaders = load_data(dataset="SkinCancer", batch_size=batch_size)
num_classes = len(dataloaders["train"].dataset.classes)
model = setup_model(model_name="mobilenet_v2", pretrained_weights=None, num_classes=num_classes)

# model.eval()
example_inputs = next(iter(dataloaders["train"]))[0].to("cpu")
# exported_model = capture_pre_autograd_graph(model, (example_inputs,))

student_model = quantization_mode(model, "export", example_inputs=(example_inputs,)).to(device)

# quantized_model = quantize_pytorch_export_model(student_model, None)

# Now load the state dict.
state_dict = torch.load(pretrained_weights, weights_only=True, map_location="cpu")
student_model.load_state_dict(state_dict)

quantized_model = quantize_pytorch_model(student_model, "export", None)

# Set to eval mode.
# torch.ao.quantization.move_exported_model_to_eval(quantized_model)

Model prepared using Export Mode QAT.


/home/jacob-delgado/anaconda3/envs/executorch/lib/python3.10/site-packages/torch/fx/graph.py:1199: UserWarning: erase_node(batch_norm_104) on an already erased node
  warnings.warn(f"erase_node({to_erase}) on an already erased node")
/home/jacob-delgado/anaconda3/envs/executorch/lib/python3.10/site-packages/torch/fx/graph.py:1199: UserWarning: erase_node(batch_norm_105) on an already erased node
  warnings.warn(f"erase_node({to_erase}) on an already erased node")
/home/jacob-delgado/anaconda3/envs/executorch/lib/python3.10/site-packages/torch/fx/graph.py:1199: UserWarning: erase_node(batch_norm_106) on an already erased node
  warnings.warn(f"erase_node({to_erase}) on an already erased node")
/home/jacob-delgado/anaconda3/envs/executorch/lib/python3.10/site-packages/torch/fx/graph.py:1199: UserWarning: erase_node(batch_norm_107) on an already erased node
  warnings.warn(f"erase_node({to_erase}) on an already erased node")
/home/jacob-delgado/anaconda3/envs/executorch/lib/python3.10/sit

In [16]:
# executorch_program = convert_to_executorch_program(quantized_model, (example_inputs,), verbose=False)
convert_to_edge(quantized_model, (example_inputs,))

Executorch Size: 9.038136


In [8]:
import os
print("Executorch:", os.path.getsize("test_quantized.pte") / 1e6)

Executorch: 9.038136
